# COMP7103 Lab 1 · Audit an AI analyst

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GITHUB_USER/comp7103-labs/blob/main/lab1/Lab1_audit.ipynb)

An AI agent explored a customer dataset and wrote a report. The dataset hides **three planted problems**. Your job is to check the agent's claims, not to redo its work.

| Time | What you do |
|---|---|
| 5 min | Read the agent's report and pick 5 claims |
| 30 min | Check each claim with the starter cells below |
| 10 min | Share-out |

This lab is ungraded practice for the AI-use log (Part C) of the assignments.

**Using this notebook in Colab.** Choose *File → Save a copy in Drive* first, and work in your copy. Run a cell with **Shift + Enter**. Run the cells from top to bottom. Where you see `___`, replace it with your own code, then run the cell again.

## 0 · Setup

This cell loads the data and the data dictionary.

In [ ]:
import os, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

# A blank (___) that you have not filled in stops with: name '___' is not defined
get_ipython().displayhook.cache_size = 0
globals().pop("___", None)

REPO = "https://raw.githubusercontent.com/doubledaibo/COMP7103B/labs/main/lab1/data/"

def get(name):
    """Use the local file if it is there (the Moodle zip), otherwise download it."""
    for path in [name, os.path.join("data", name),]:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    try:
        urllib.request.urlretrieve(REPO + name, os.path.join("data", name))
    except Exception:
        raise SystemExit(f"Could not download {name}. Upload it with the folder icon on the left, "
                         "or download the Lab 1 zip from Moodle.")
    return os.path.join("data", name)

df = pd.read_csv(get("customer_churn.csv"))
dictionary = pd.read_csv(get("data_dictionary.csv"))
print(df.shape)
df.head()

## 1 · The agent's report (5 min)

### The agent's report

**Task given to the agent:** "Explore this dataset and report the most important findings."

*Prepared for COMP7103 Lab 1. The agent's code in the log really ran on this data, and its outputs are shown unchanged. The report reproduces mistakes that AI agents commonly make.*

#### Findings

1. The dataset covers 3,000 customers. *(see Step 1)*
2. The average customer is 43.4 years old. *(see Step 3)*
3. No column has missing values, so no cleaning is needed. *(see Step 2)*
4. Support calls are a strong warning sign: churn rises from 12.9% for customers with 0–1 calls to 63.9% for 4 or more calls. *(see Step 5)*
5. Monthly-contract customers churn about 4 times as often as yearly ones (30.9% vs 7.8%). *(see Step 4)*
6. A gradient boosting model predicts churn with 97.7% test accuracy. The most important feature is cancellation_fee_charged, so the cancellation fee policy drives churn. *(see Step 7)*
7. Tenure ranges from 0 to 240 months. The 119 customers with more than 10 years of tenure have a churn rate of 5%: they are the most loyal group. *(see Step 6)*
8. Monthly spend ranges from HK$38 to HK$480 and shows no unusual values. *(see Step 3)*
9. Customers who signed up online churn about 40% less than customers who signed up in a shop. *(see Step 4)*

#### Recommendation

Target retention offers at monthly-contract customers with many support calls, and review the cancellation fee policy, which the model identifies as the main driver of churn.

The agent's code and outputs are in **the agent's log** at the end of this notebook.

**Your task:** pick 5 claims. Try to cover all three planted problems. For each claim, ask: did the agent compute this number, and is the data behind it sound?

## 2 · Guided checks (30 min)

Use the checks you need for your 5 claims. Each check matches an item of the *Verify, don't trust* checklist from the lecture.

### Check A · Count the customers

*Recompute the key numbers.* Does every row belong to a different customer?

In [ ]:
print("rows:", len(df))
print("unique customers:", df["customer_id"].___())

In [ ]:
# Show all rows of one customer who appears more than once
repeated = df.loc[df["customer_id"].duplicated(), "customer_id"]
print(len(repeated), "repeated rows")
df[df["customer_id"] == repeated.iloc[0]]

### Check B · Look at the smallest and largest values

*Mind the data.* Compare each min and max with the rules in the data dictionary.

In [ ]:
dictionary

In [ ]:
df.describe().round(1)

In [ ]:
# How many rows break each rule? Fill in the limits from the data dictionary.
print("age below 18:", (df["age"] < ___).sum())
print("age above 100:", (df["age"] > 100).sum())
print("tenure longer than the company has existed:", (df["tenure_months"] > ___).sum())

In [ ]:
df["tenure_months"].plot.hist(bins=48, title="tenure_months")
plt.show()

### Check C · Missing values in disguise

`df.isna()` only finds empty cells. Codes such as -1 or 999 also mean "missing".

In [ ]:
df.isna().sum()   # what the agent ran

In [ ]:
# The smallest values of age, and how often each appears
df["age"].value_counts().sort_index().head()

In [ ]:
# The largest values of support_calls_90d. Fill in: head() or tail()?
df["support_calls_90d"].value_counts().sort_index().___()

### Check D · Recompute a rate by group

*Recompute* the agent's group rates. Which column defines the groups?

In [ ]:
df.groupby("___")["churned"].mean().round(3)

### Check E · Hunt for leakage

Read the `when_recorded` column of the data dictionary. Which column is only known **after** the outcome? Then *compare to a baseline*: how accurate is the rule "nobody leaves"?

In [ ]:
pd.crosstab(df["cancellation_fee_charged"], df["churned"])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

def accuracy(data, drop=()):
    """The agent's model, trained without the columns in drop."""
    X = pd.get_dummies(data.drop(columns=["customer_id", "churned", *drop]), drop_first=True)
    y = data["churned"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
    model = GradientBoostingClassifier(random_state=0).fit(X_train, y_train)
    return model.score(X_test, y_test), 1 - y_test.mean()

acc, baseline = accuracy(df, drop=["___"])
print(f"model accuracy {acc:.3f}   baseline 'nobody leaves' {baseline:.3f}")

### Check F · Clean, then recompute

Make a cleaned copy: one row per customer, and impossible values set to missing (NaN). Then re-run any check above on `clean` instead of `df`.

In [ ]:
clean = df.drop_duplicates(subset="___").copy()   # one row per customer
clean.loc[(clean["age"] < 18) | (clean["age"] > 100), "age"] = float("nan")
clean.loc[clean["tenure_months"] > 137, "tenure_months"] = float("nan")
clean.loc[clean["support_calls_90d"] == ___, "support_calls_90d"] = float("nan")
print(len(clean), "customers")

In [ ]:
# Example: the agent's Step 4 again, on the cleaned data
print(clean["churned"].mean().round(3))
clean.groupby("contract")["churned"].mean().round(3)

In [ ]:
# Your turn: recompute another claim on the cleaned data


## 3 · Your audit table

Double-click this cell to edit it. Verdicts: **Correct**, **Partly correct**, **Wrong**, **Cannot tell yet**.

| # | Agent's claim | Your check (what you ran or read) | Verdict |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |
| 4 |  |  |  |
| 5 |  |  |  |

**Planted problems found** (replace [ ] with [x]):

- [ ] duplicates
- [ ] impossible values
- [ ] a leaky column

## 4 · Share-out (10 min)

- Pick one claim: what did the agent say, what did you check, what is your verdict?
- Which planted problems did the agent miss?
- What would you change in the agent's recommendation?

## Optional · Audit your own AI tool

This dataset is synthetic, so you may give it to an AI tool. Ask the same question as the agent: *"Explore this dataset and report the most important findings."* Audit its answer with the checks above. Write down the tool, your prompt and your verdicts, in the same format as the AI-use log of the assignments.

# Appendix · The agent's log

The code the agent ran on `customer_churn.csv`, and what it printed.

## Step 1 · Load the data

```python
import pandas as pd
df = pd.read_csv("customer_churn.csv")
print(df.shape)
print(df.head())
```

Output:

```
(3000, 8)
  customer_id  age contract  tenure_months  monthly_spend  support_calls_90d  cancellation_fee_charged  churned
0      C39048   34  monthly             48            295                  1                         0        0
1      C63693   20   yearly             57            190                  0                         0        0
2      C71090   41   yearly              6            211                  1                         0        0
3      C49809   59  monthly             16            178                  0                         0        0
4      C49061   24  monthly              8            370                  1                         1        1
```

## Step 2 · Check for missing values

```python
print(df.isna().sum())
```

Output:

```
customer_id                 0
age                         0
contract                    0
tenure_months               0
monthly_spend               0
support_calls_90d           0
cancellation_fee_charged    0
churned                     0
dtype: int64
```

## Step 3 · Summary statistics

```python
print(df.describe().round(1))
```

Output:

```
          age  tenure_months  monthly_spend  support_calls_90d  cancellation_fee_charged  churned
count  3000.0         3000.0         3000.0             3000.0                    3000.0   3000.0
mean     43.4           32.0          210.8                9.4                       0.2      0.2
std      14.8           33.3           93.5               90.7                       0.4      0.4
min      -1.0            0.0           38.0                0.0                       0.0      0.0
25%      34.0            8.0          142.0                0.0                       0.0      0.0
50%      44.0           21.0          192.0                1.0                       0.0      0.0
75%      53.0           44.0          261.0                2.0                       0.0      0.0
max     199.0          240.0          480.0              999.0                       1.0      1.0
```

## Step 4 · Churn by contract type

```python
print(df["churned"].mean().round(3))
print(df.groupby("contract")["churned"].mean().round(3))
```

Output:

```
0.206
contract
monthly    0.309
yearly     0.078
Name: churned, dtype: float64
```

## Step 5 · Churn by support calls

```python
calls = pd.cut(df["support_calls_90d"], [-1, 1, 3, 10_000], labels=["0-1", "2-3", "4+"])
print(df.groupby(calls, observed=True)["churned"].agg(["mean", "size"]).round(3))
```

Output:

```
                    mean  size
support_calls_90d             
0-1                0.129  2074
2-3                0.337   793
4+                 0.639   133
```

## Step 6 · Tenure

```python
print(df["tenure_months"].agg(["min", "max"]))
print(df.loc[df["tenure_months"] > 120, "churned"].agg(["mean", "size"]))
```

Output:

```
min      0
max    240
Name: tenure_months, dtype: int64
mean      0.05042
size    119.00000
Name: churned, dtype: float64
```

## Step 7 · Churn model

```python
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
X = pd.get_dummies(df.drop(columns=["customer_id", "churned"]), drop_first=True)
y = df["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
model = GradientBoostingClassifier(random_state=0).fit(X_train, y_train)
print("test accuracy:", round(model.score(X_test, y_test), 3))
print(pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).round(3))
```

Output:

```
test accuracy: 0.977
cancellation_fee_charged    0.955
monthly_spend               0.015
support_calls_90d           0.010
tenure_months               0.008
age                         0.008
contract_yearly             0.004
dtype: float64
```